In [1]:
import os
import pandas as pd
import numpy as np
from slugify import slugify

In [2]:
results_segment_file = r'D:\Projects\corrosions\tests\results_segment.xlsx'
label_file = r'D:\Projects\corrosions\tests\labels.xlsx'
json_dir = r"D:\Projects\pgn-laravel\storage\app\json"

In [3]:
df = pd.read_excel(label_file)

In [4]:
def replace_excel_file(value):
    if value is None:
        return None
    if value is np.nan:
        return None
    return os.path.basename(value).replace('.xlsx', '.json')

In [5]:
df['normalized_acvg_dcvg_file'] = df['normalized_acvg_dcvg_file'].apply(replace_excel_file)
df['normalized_cips_file'] = df['normalized_cips_file'].apply(replace_excel_file)
df['normalized_pcm_file'] = df['normalized_pcm_file'].apply(replace_excel_file)

In [6]:
df.to_json(os.path.join(json_dir, 'label.json'), orient='records')

### Build Seeder Updater
Make surre to run `calculate-stats.ipynb`

In [27]:
segment_df = pd.read_excel(results_segment_file)

In [9]:
def get_province(area):
    if area == 'Tangerang' or area == 'Cilegon':
        return '36'
    if area == 'Jakarta':
        return '31'
    return '32'

#### Build Area updater

In [10]:
area_df = segment_df.groupby(['Year', 'Area'])

In [11]:
area_sum_df = area_df.sum()

In [12]:
area_sum_df.drop(columns=['Area Code','Jalur','Segment Code','Diameter (inch)','CIPS Protection'], inplace=True)

In [13]:
area_sum_df

Panjang (km)  Protected  Unprotected  Medium to Poor  \
Year Area                                                              
2024 Bekasi            35.74     156.05       243.95          140.84   
     Bogor              9.95      93.52         6.48           53.38   
     Cilegon           37.89     256.35       143.65          134.31   
     Cirebon           18.75     157.86       342.14          201.51   
     Jakarta           24.50     131.57       168.43          125.36   
     Karawang          16.92      90.84         9.16           28.44   
     Tangerang         76.69     309.77        90.23          122.02   
2025 Bekasi            28.00     243.28       256.72          220.20   
     Bogor             46.30     390.60       409.40          381.66   
     Cilegon           20.53     936.55       163.45          503.35   
     Cirebon           33.17     287.61       212.39          154.11   
     Jakarta           38.82     740.17       459.83          584.15   
     Karawang          41.00     775.92       124.08          338.25   
     Tangerang         32.19    1632.09       667.91          872.79   

                Medium to High  Total Anomali  
Year Area                                      
2024 Bekasi             259.16             12  
     Bogor               46.62              8  
     Cilegon            265.69              0  
     Cirebon            298.49             13  
     Jakarta            174.64             32  
     Karawang            71.56              3  
     Tangerang          277.98             31  
2025 Bekasi             279.80             58  
     Bogor              418.34             48  
     Cilegon            596.65              2  
     Cirebon            345.89             32  
     Jakarta            615.85             85  
     Karawang           561.75             46  
     Tangerang         1427.21            130

In [14]:
area = []

for idx in area_sum_df.index:
    year, area_idx = idx
    row = area_sum_df.loc[idx]

    total_condition = row['Protected'] + row['Unprotected']
    percent_protected = row['Protected'] / total_condition * 100
    percent_unprotected = row['Unprotected'] / total_condition * 100

    total_quality = row['Medium to Poor'] + row['Medium to High']
    percent_medium_to_poor = row['Medium to Poor'] / total_quality * 100
    percent_medium_to_high = row['Medium to High'] / total_quality * 100

    _area = {
        'name': area_idx,
        'code': slugify(f"{area_idx}-{year}"),
        'year': year,
        'total_length': row['Panjang (km)'],
        'protected' : round(percent_protected, 2),
        'unprotected' : round(percent_unprotected, 2),
        'medium_to_poor' : round(percent_medium_to_poor, 2),
        'medium_to_high' : round(percent_medium_to_high, 2),
        'total_anomaly' : row['Total Anomali'],
        'province_code': get_province(area_idx),
        'created_by': 'test@test.com',
    }

    area.append(_area)

In [15]:
area_df = pd.DataFrame(area)
area_df.to_json(os.path.join(json_dir, 'area.json'), orient='records')

#### Build Segment Seeder

In [28]:
segment_df

,Year,Area,Area Code,Jalur,Segment Code,Diameter (inch),CIPS Protection,Panjang (km),Protected,Unprotected,Medium to Poor,Medium to High,Total Anomali
0,2025,Jakarta,jakarta-2025,Pipa Servis Indonesia Power,pipa-servis-indonesia-power-16,16,SACP,1.75,100.00,0.00,65.57,34.43,3
1,2025,Jakarta,jakarta-2025,RE Martadinata - Jl. Industri Salim Ivomas 2,re-martadinata-jl-industri-salim-ivomas-2-16,16,SACP,1.70,100.00,0.00,49.12,50.88,2
2,2025,Jakarta,jakarta-2025,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,jl-ps-minggu-spbg-perumahan-koperasi-jl-g-subr...,10,SACP,3.67,0.44,99.56,57.89,42.11,2
3,2025,Jakarta,jakarta-2025,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,outlet-mrs-pondok-ungu-1-reducer-pipa-8-tee-va...,10,SACP,0.84,100.00,0.00,43.18,56.82,2
4,2025,Jakarta,jakarta-2025,Parang Tritis - Ancol,parang-tritis-ancol-10,10,SACP,1.51,32.46,67.54,32.76,67.24,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,2024,Cirebon,cirebon-2024,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,std-garawangi-sungai-cipetir-selatan-bv-cilump...,6,SACP,5.32,100.00,0.00,27.79,72.21,3
91,2024,Cilegon,cilegon-2024,Cilegon - Merak (SV 06 Grogol - SV 07),cilegon-merak-sv-06-grogol-sv-07-16,16,ICCP,4.94,100.00,0.00,35.64,64.36,0
92,2024,Cilegon,cilegon-2024,Bojonegara - Suralaya (SV 01 - SV 04),bojonegara-suralaya-sv-01-sv-04-16,16,ICCP,16.53,85.02,14.98,40.02,59.98,0
93,2024,Cilegon,cilegon-2024,Cilegon - Anyer (SV 05 - SV 14),cilegon-anyer-sv-05-sv-14-16,16,ICCP,13.30,71.33,28.67,33.85,66.15,0


In [44]:
segments = []

for index in segment_df.index:
    row = segment_df.iloc[index]

    _segment = {
        'name': row['Jalur'],
        'code': row['Segment Code'],
        'diameter': row['Diameter (inch)'],
        'year': row['Year'],
        'area_code': row['Area Code'],
        'pipe_length': row['Panjang (km)'],
        'cips_protection': row['CIPS Protection'],
        'protected' : row['Protected'],
        'unprotected' : row['Unprotected'],
        'medium_to_poor' : row['Medium to Poor'],
        'medium_to_high' : row['Medium to High'],
        'total_anomaly' : row['Total Anomali'],
        'province_code': get_province(row['Area']),
        'acvg_dcvg_normalized_file': df[df['segment_code'] == row['Segment Code']]['normalized_acvg_dcvg_file'].iloc[0],
        'cips_normalized_file': df[df['segment_code'] == row['Segment Code']]['normalized_cips_file'].iloc[0],
        'pcm_normalized_file': df[df['segment_code'] == row['Segment Code']]['normalized_pcm_file'].iloc[0],
        'created_by': 'test@test.com',
    }

    segments.append(_segment)

In [45]:
segments_df = pd.DataFrame(segments)
segments_df.to_json(os.path.join(json_dir, 'segments.json'), orient='records')

In [46]:
segments_df

,name,code,diameter,year,area_code,pipe_length,cips_protection,protected,unprotected,medium_to_poor,medium_to_high,total_anomaly,province_code,acvg_dcvg_normalized_file,cips_normalized_file,pcm_normalized_file,created_by
0,Pipa Servis Indonesia Power,pipa-servis-indonesia-power-16,16,2025,jakarta-2025,1.75,SACP,100.00,0.00,65.57,34.43,3,31,acvg-dcvg-pipa-servis-indonesia-power-16-jakar...,cips-sacp-01-jkt-16-in-pipa-servis-indonesia-p...,pcm-01-jkt-pipa-servis-16-indonesia-power-data...,test@test.com
1,RE Martadinata - Jl. Industri Salim Ivomas 2,re-martadinata-jl-industri-salim-ivomas-2-16,16,2025,jakarta-2025,1.70,SACP,100.00,0.00,49.12,50.88,2,31,acvg-dcvg-re-martadinata-jl-industri-salim-ivo...,cips-sacp-02-jkt-16-in-re-martadinata-jl-indus...,pcm-02-jkt-invomas-data.json,test@test.com
2,Jl. Ps. Minggu/ SPBG - Perumahan Koperasi/Jl. ...,jl-ps-minggu-spbg-perumahan-koperasi-jl-g-subr...,10,2025,jakarta-2025,3.67,SACP,0.44,99.56,57.89,42.11,2,31,acvg-dcvg-jl-ps-minggu-spbg-perumahan-koperasi...,cips-sacp-03-jkt-10-in-jl-ps-minggu-spbg-perum...,pcm-03-jkt-jl-ps-minggu-spbg-perumahan-koperas...,test@test.com
3,outlet MRS Pondok Ungu 1 Reducer Pipa 8'' - Te...,outlet-mrs-pondok-ungu-1-reducer-pipa-8-tee-va...,10,2025,jakarta-2025,0.84,SACP,100.00,0.00,43.18,56.82,2,31,acvg-dcvg-mrs-pondok-ungu-1-reducer-pipa-8-tee...,cips-sacp-04-jkt-10-in-outlet-mrs-pondok-ungu-...,pcm-04-jkt-mrs-pondok-ungu-harapan-indah-data....,test@test.com
4,Parang Tritis - Ancol,parang-tritis-ancol-10,10,2025,jakarta-2025,1.51,SACP,32.46,67.54,32.76,67.24,8,31,acvg-dcvg-parang-tritis-ancol-10-jakarta-sheet...,cips-sacp-05-jkt-10-in-parang-tritis-ancol-dat...,pcm-05-jkt-parang-tritis-ancol-data.json,test@test.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,STD Garawangi - Sungai Cipetir Selatan - BV Ci...,std-garawangi-sungai-cipetir-selatan-bv-cilump...,6,2024,cirebon-2024,5.32,SACP,100.00,0.00,27.79,72.21,3,32,acvg-dcvg-std-garawangi-sungai-cipetir-selatan...,cips-sacp-segmen-crb-6-inch-std-garawangi-sung...,pcm-crb-6-inch-std-garawangi-sungai-cipetir-se...,test@test.com
91,Cilegon - Merak (SV 06 Grogol - SV 07),cilegon-merak-sv-06-grogol-sv-07-16,16,2024,cilegon-2024,4.94,ICCP,100.00,0.00,35.64,64.36,0,36,None,cips-iccp-segmen-clg-16-inch-cilegon-merak-sv0...,pcm-clg-16-inch-merak-cilegon-grogol-sequentia...,test@test.com
92,Bojonegara - Suralaya (SV 01 - SV 04),bojonegara-suralaya-sv-01-sv-04-16,16,2024,cilegon-2024,16.53,ICCP,85.02,14.98,40.02,59.98,0,36,None,cips-iccp-segmen-clg-16-inch-suralaya-bojonega...,pcm-clg-16-inch-bojonegara-suralaya-sv-01-sv-0...,test@test.com
93,Cilegon - Anyer (SV 05 - SV 14),cilegon-anyer-sv-05-sv-14-16,16,2024,cilegon-2024,13.30,ICCP,71.33,28.67,33.85,66.15,0,36,None,cips-iccp-segmen-clg-16-inch-cilegon-anyer-sv0...,pcm-clg-16-inch-anyer-cilegon-anyer-cilegon.json,test@test.com
